In [3]:
import pandas as pd
import numpy as np

# CONFIG
PESO_PAO = 0.064
PAES_POR_TELA = 25

# Ler arquivo
df = pd.read_excel(r"C:\Users\Edgar.souza\Downloads\pao_frances_base.xlsx")

# Ajustar data
df['Data'] = pd.to_datetime(df['Data'])

# Hora início
df['Hora_inicio'] = df['Faixa Hora N1'].str.split(' - ').str[0]
df['Hora'] = pd.to_datetime(df['Hora_inicio'], format='%H:%M')

# Traduzir dias
dias_pt = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['Dia_Semana'] = df['Data'].dt.day_name().map(dias_pt)

# Criar HORA cheia (base da venda)
df['Hora_cheia'] = df['Hora'].dt.floor('1h')

# AGRUPAR POR HORA (PICO)
agrupado = df.groupby(
    ['Unidade Resumo', 'Dia_Semana', 'Hora_cheia']
)['Qtde. Venda'].mean().reset_index()

# Converter para pães
agrupado['Qtd_Paes'] = np.ceil(agrupado['Qtde. Venda'] / PESO_PAO)

# Converter para telas
agrupado['Qtd_Telas'] = np.ceil(agrupado['Qtd_Paes'] / PAES_POR_TELA)

# 🔥 DIVIDIR EM 30 MINUTOS
linhas = []

for _, row in agrupado.iterrows():
    total = int(row['Qtd_Telas'])
    
    primeira_metade = int(np.ceil(total / 2))
    segunda_metade = int(np.floor(total / 2))
    
    hora_base = row['Hora_cheia']
    
    # 00 min
    linhas.append({
        'Unidade Resumo': row['Unidade Resumo'],
        'Dia_Semana': row['Dia_Semana'],
        'Faixa_30min': hora_base.strftime('%H:%M'),
        'Qtd_Telas': primeira_metade
    })
    
    # 30 min
    linhas.append({
        'Unidade Resumo': row['Unidade Resumo'],
        'Dia_Semana': row['Dia_Semana'],
        'Faixa_30min': (hora_base + pd.Timedelta(minutes=30)).strftime('%H:%M'),
        'Qtd_Telas': segunda_metade
    })

final = pd.DataFrame(linhas)

# Ordenar dias
ordem_dias = [
    'Segunda-feira', 'Terça-feira', 'Quarta-feira',
    'Quinta-feira', 'Sexta-feira', 'Sábado', 'Domingo'
]

final['Dia_Semana'] = pd.Categorical(
    final['Dia_Semana'],
    categories=ordem_dias,
    ordered=True
)

final = final.sort_values(
    ['Unidade Resumo', 'Dia_Semana', 'Faixa_30min']
)

# Exportar por loja
for loja in final['Unidade Resumo'].unique():
    dados_loja = final[final['Unidade Resumo'] == loja]
    dados_loja.to_excel(f"{loja}.xlsx", index=False)

print("Arquivos gerados com divisão por 30 minutos!")

Arquivos gerados com divisão por 30 minutos!
